<a href="https://colab.research.google.com/github/Ayu-sshhhhh/NLP-by-me/blob/main/Into_to_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP Fundamentals
NLP has the goal of deriving info. from natural language (could be texts or sequence).
Another common term for NLP problems is sequence to sequence problems (seq2seq).

# Getting some pre-defined helper functions


In [37]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

--2026-06-30 07:38:41--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py.1’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-06-30 07:38:41 (44.1 MB/s) - ‘helper_functions.py.1’ saved [10246/10246]



# Get a text dataset
The dataset we're going to be using is Kaggle's introduction to NLP
See the original source here : https://www.kaggle.com/c/nlp-getting-started

In [38]:
import pandas as pd
from pandas import read_csv
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

### Glimpse of Data

In [39]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [40]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [41]:
test_df.tail()

,id,keyword,location,text
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...
3262,10875,NaN,NaN,#CityofCalgary has activated its Municipal Eme...


In [42]:
len(test_df), len(train_df)

(3263, 7613)

In [43]:
# Shuffling the data (just for fun)
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


# Split data into train and validation sets

In [44]:
from sklearn.model_selection import train_test_split

In [45]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)


In [46]:
len(train_sentences)

6851

In [47]:
len(val_sentences)

762

In [48]:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Convert texts to numbers

In [49]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# This is a sample TextVectorizzation object
text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode="int",
                                    output_sequence_length=None)

In [50]:
max_vocab_length=10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [51]:
text_vectorizer.adapt(train_sentences)

In [52]:
sample_sentence = "My name is Ana, I am from England. Nice to meet you !"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[  13,  735,    9,    1,    8,  160,   20, 2126, 1117,    5, 1389,
          12,    0,    0,    0]])>

### Tokenizing a random sentence

In [53]:
import random
random_sentence = random.choice(train_sentences)
print(f"Original Sentence : \n{random_sentence}\
\n\nVectorized version :")
text_vectorizer([random_sentence])

Original Sentence : 
@greateranglia I know the cow incident not yr fault by how where they on the line could of caused a derailment

Vectorized version :


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1,    8,  106,    2, 5900, 1656,   34, 2591,    1,   18,   62,
         221,   64,   11,    2]])>

### Unique words in vocab

Here, [UNK] token is used for 'unknown' words

In [54]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f"Number of words in vocab : {len(words_in_vocab)}")
print(f"Top 5 words in vocab : {top_5_words}")
print(f"Bottom 5 words in vocab : {bottom_5_words}")

Number of words in vocab : 10000
Top 5 words in vocab : ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab : [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


In [55]:
words_in_vocab[12:15]

[np.str_('you'), np.str_('my'), np.str_('with')]

# Cooking an Embedding layer

In [56]:
tf.random.set_seed(42)
from tensorflow.keras import layers
embedding = layers.Embedding(input_dim=max_vocab_length,
                             output_dim = 128,
                             embeddings_initializer="uniform",
                             name="embedding_1")
embedding

<Embedding name=embedding_1, built=False>

As seen above, `built=False` means :
Keras uses a *lazy initialization* strategy. When you define layers.Embedding(...), Keras registers the configuration you want (like the vocabulary size and output dimensions), but it does not actually create the weight matrices in memory yet.

It waits to officially "build" the layer and allocate memory until one of two things happens:

* You pass some actual data through the layer for the first time.
* You manually call the .build(input_shape) method.

In [57]:
sample_embed = embedding(text_vectorizer(["Hehe, haha, huhu"]))
sample_embed

<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.01199991, -0.03446921,  0.01478753, ...,  0.02665604,
          0.00051999, -0.02201831],
        [ 0.03191612,  0.02555032, -0.04591912, ..., -0.04704391,
          0.01240655, -0.03842626],
        [-0.01199991, -0.03446921,  0.01478753, ...,  0.02665604,
          0.00051999, -0.02201831],
        ...,
        [ 0.01491112, -0.00431412,  0.01399591, ..., -0.02069417,
          0.02173979, -0.00708868],
        [ 0.01491112, -0.00431412,  0.01399591, ..., -0.02069417,
          0.02173979, -0.00708868],
        [ 0.01491112, -0.00431412,  0.01399591, ..., -0.02069417,
          0.02173979, -0.00708868]]], dtype=float32)>

In [58]:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([-0.01199991, -0.03446921,  0.01478753,  0.02679098, -0.00750031,
       -0.02766027, -0.01503419,  0.01470778, -0.0350918 , -0.04608241,
       -0.00215838, -0.00871484,  0.04544702, -0.00311419, -0.04630068,
       -0.02247217,  0.02050615, -0.01371501, -0.01626183,  0.04775662,
       -0.01449375, -0.02256127,  0.03075432, -0.00343517, -0.03676372,
        0.04121419,  0.02939589,  0.02023616, -0.02396101, -0.01643267,
        0.04094351, -0.00317782,  0.03722311, -0.04290178, -0.00575777,
       -0.03799816,  0.04421416, -0.00493119,  0.00945673,  0.04266027,
        0.04895974,  0.00199478, -0.03652341, -0.02360644,  0.03480102,
       -0.03847434, -0.04174332, -0.00175717,  0.03385527,  0.02095175,
       -0.00277219,  0.03171095,  0.00883268, -0.0062741 , -0.03430518,
        0.02339444,  0.04310173,  0.049002  , -0.02211989,  0.04423393,
       -0.02340095, -0.03869401, -0.03756828,  0.01519766, -0.00240647,
       -0.003368

# Modelling

* **Model 0** : Naive Bayes (baseline)
* **Model 1** : Feed-forward NN (dense model)
* **Model 2** : LSTM model
* **Model 3** : GRU model
* **Model 4** : Bidirectional-LSTM model
* **Modle 5** : 1D Conv NN
* **Model 6** : TFHub pre-trained feature extractor
* **Model 7** : Same as model 6 with 10 % of training data

## Model 0

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [60]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of : {baseline_score*100:.2f} %")

Our baseline model achieves an accuracy of : 79.27 %


In [63]:
# Making predictions with our baseline model

baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1])

In [71]:
# Function to evaluate : accuracy, precision, recall, f1-score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of 1D array

  Returns a dict of accuracy, precision, recall, f1-score.
  """
  model_accuracy = accuracy_score(y_true, y_pred) *100
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy" : model_accuracy,
                   "precision" : model_precision,
                   "recall" : model_recall,
                   "f1" : model_f1}
  return model_results

In [72]:
baseline_results = calculate_results(y_true = val_labels,
                                     y_pred = baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

## Model 1

In [73]:
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_1 = tf.keras.Model(inputs,outputs, model="model_1_dense")

In [74]:
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [75]:
model_1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_4            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [76]:
model_1_history = model_1.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - accuracy: 0.6926 - loss: 0.6075 - val_accuracy: 0.7559 - val_loss: 0.5343
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8159 - loss: 0.4434 - val_accuracy: 0.7861 - val_loss: 0.4738
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8574 - loss: 0.3509 - val_accuracy: 0.7953 - val_loss: 0.4616
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.8888 - loss: 0.2885 - val_accuracy: 0.7874 - val_loss: 0.4676
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9092 - loss: 0.2417 - val_accuracy: 0.7808 - val_loss: 0.4832


In [77]:
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7808 - loss: 0.4832


[0.483174592256546, 0.7808399200439453]

In [78]:
model_1_pred_probs = model_1.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [80]:
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [82]:
model_1_results = calculate_results(y_true=val_labels,
                                  y_pred=model_1_preds)
model_1_results

{'accuracy': 78.08398950131233,
 'precision': 0.783783808499639,
 'recall': 0.7808398950131233,
 'f1': 0.7783998521836788}

##### Another helper Func.

In [85]:
def compare_baseline_to_new_results(baseline_results, new_model_results):
  for key, value in baseline_results.items():
    print(f"Baseline {key}: {value:.2f}, New {key}: {new_model_results[key]:.2f}, Difference {new_model_results[key]-value:.2f}")
compare_baseline_to_new_results(baseline_results=baseline_results,
                                new_model_results=model_1_results)

Baseline accuracy: 79.27, New accuracy: 78.08, Difference -1.18
Baseline precision: 0.81, New precision: 0.78, Difference -0.03
Baseline recall: 0.79, New recall: 0.78, Difference -0.01
Baseline f1: 0.79, New f1: 0.78, Difference -0.01


## Model 2

In [94]:
tf.random.set_seed(42)

from tensorflow.keras import layers
model_2_embedding = layers.Embedding(input_dim = max_vocab_length,
                                    output_dim = 128,
                                    embeddings_initializer = "uniform",
                                    name="embedding_2")
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_2_embedding(x)
print(x.shape)
x = layers.LSTM(64)(x)
print(x.shape)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

(None, 15, 128)
(None, 64)


In [95]:
model_2.compile(loss="binary_crossentropy",
                optimizer = "Adam",
                metrics = ["accuracy"])

In [96]:
model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_4            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [97]:
model_2_history = model_2.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 15s 45ms/step - accuracy: 0.7425 - loss: 0.5110 - val_accuracy: 0.7782 - val_loss: 0.4599
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.8733 - loss: 0.3123 - val_accuracy: 0.7520 - val_loss: 0.5203
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.9222 - loss: 0.2117 - val_accuracy: 0.7520 - val_loss: 0.6670
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9521 - loss: 0.1452 - val_accuracy: 0.7520 - val_loss: 0.7784
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9613 - loss: 0.1195 - val_accuracy: 0.7664 - val_loss: 0.5897


In [98]:
# Make predictions on the validation dataset
model_2_pred_probs = model_2.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step


In [100]:
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))

In [101]:
# Calculate LSTM model results
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 76.64041994750657,
 'precision': 0.7751902139142625,
 'recall': 0.7664041994750657,
 'f1': 0.761330428785352}

In [102]:
compare_baseline_to_new_results(baseline_results, model_2_results)

Baseline accuracy: 79.27, New accuracy: 76.64, Difference -2.62
Baseline precision: 0.81, New precision: 0.78, Difference -0.04
Baseline recall: 0.79, New recall: 0.77, Difference -0.03
Baseline f1: 0.79, New f1: 0.76, Difference -0.02
